In [1]:
import json
from pathlib import Path
from datetime import date

import polars as pl
import yaml

from common import read_day, daterange

with open("../config.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [ ]:
days = (
    list(daterange(date(2023, 10, 1), date(2023, 12, 31)))
    + list(daterange(date(2024, 1, 1), date(2024, 2, 29)))
    + list(daterange(date(2024, 10, 1), date(2024, 12, 31)))
    + list(daterange(date(2025, 10, 1), date(2025, 10, 31)))
)

In [ ]:
data = pl.DataFrame()
for i in days:
    temp = read_day(i, config["dirs"]["pickwell"])
    dids = temp.unique(subset=["device_aid_hashed"], keep="first").select(
        "device_aid_hashed"
    )
    data = pl.concat([data, dids])
# runs 10 minutes

In [18]:
unique_device_ids = data.unique(keep="first").sort(["device_aid_hashed"])
unique_device_ids = unique_device_ids.with_columns(
    pl.int_range(1, pl.len() + 1, dtype=pl.UInt32).alias("new")
)

In [21]:
lookup = {}
for row in unique_device_ids[["device_aid_hashed", "new"]].iter_rows(named=True):
    lookup[row["device_aid_hashed"]] = row["new"]

In [22]:
Path("../output/id_lookup").mkdir(parents=True, exist_ok=True)
with open("../output/id_lookup/id_lookup.json", "w") as fp:
    json.dump(lookup, fp, indent=1)